[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jmdvinodjmd/agentic-ai-tutorial/blob/feature/notebook-rebuild/notebooks/case_studies/research_assistant/langgraph.ipynb)

# Research assistant — LangGraph

The matched five-node graph plans, retrieves, builds a claim ledger, synthesises and critiques. It uses the same catalogue, mock decisions, four-call budget and checks as plain Python. Mock runtime is under one minute on CPU; local inference is practical but slower.

In [ ]:
import subprocess
import sys

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "langgraph==1.2.9", "pydantic==2.12.5"],
        check=True,
    )

In [ ]:
import json
from pathlib import Path
from typing import ClassVar

from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, ConfigDict, Field

ROOT = Path.cwd()
while ROOT.parent != ROOT and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from agentic_tutorial.models import DeterministicMockClient  # noqa: E402
from agentic_tutorial.models.providers.fixtures import ScriptedScenarioFixture  # noqa: E402
from agentic_tutorial.safety import (  # noqa: E402
    PolicyOutcome,
    SafetyEngine,
    SafetyPolicy,
    UntrustedContent,
)
from agentic_tutorial.schemas import CriticDecision, Message, MessageRole  # noqa: E402

catalogue = json.loads((ROOT / "data/research_assistant/evidence_catalogue.json").read_text())
fixture = json.loads((ROOT / "data/research_assistant/case_mock.json").read_text())

## Contracts and graph nodes

Model proposals cross typed boundaries. Retrieved passages remain untrusted; injection indicators remove a record before extraction. Each node returns explicit state, trace and usage.

In [ ]:
class Strict(BaseModel):
    model_config = ConfigDict(extra="forbid")


class QueryPlan(Strict):
    schema_id: ClassVar[str] = "research.query.v1"
    queries: tuple[str, ...] = Field(min_length=1, max_length=4)


class Claim(Strict):
    source_id: str
    claim: str
    stance: str


class Ledger(Strict):
    schema_id: ClassVar[str] = "research.ledger.v1"
    claims: tuple[Claim, ...]


class Synthesis(Strict):
    schema_id: ClassVar[str] = "research.synthesis.v1"
    answer: str
    source_ids: tuple[str, ...]
    outcome: str


Ledger.model_rebuild(_types_namespace={"Claim": Claim})


def fresh_model():
    return DeterministicMockClient(ScriptedScenarioFixture.model_validate(fixture))


def user(text):
    return Message(role=MessageRole.USER, content=text)


async def propose(client, schema, prompt):
    response = await client.generate([user(prompt)], response_schema=schema)
    return schema.model_validate(response.structured_output)


def search(query):
    terms = set(query.casefold().split())
    return [
        record
        for record in catalogue["records"]
        if terms & set((record["title"] + " " + record["passage"]).casefold().split())
    ]

In [ ]:
def build_graph():
    client = fresh_model()
    safety = SafetyEngine(SafetyPolicy(allowed_tools=("catalogue_search",)))

    async def plan_node(state):
        plan = await propose(client, QueryPlan, "Plan bounded searches.")
        return {
            "plan": plan,
            "trace": [{"event": "plan", "queries": plan.queries}],
            "model_calls": 1,
        }

    def retrieve_node(state):
        retrieved = {
            record["source_id"]: record
            for query in state["plan"].queries
            for record in search(query)
        }
        assessments = {
            sid: safety.inspect_retrieved_content(
                UntrustedContent(source_id=sid, text=record["passage"])
            )
            for sid, record in retrieved.items()
        }
        safe = {
            sid: record
            for sid, record in retrieved.items()
            if assessments[sid].decision.outcome in {PolicyOutcome.ALLOW, PolicyOutcome.TRANSFORM}
            and not assessments[sid].indicators
        }
        return {
            **state,
            "safe": safe,
            "trace": [
                *state["trace"],
                {
                    "event": "retrieve",
                    "source_ids": sorted(retrieved),
                    "isolated": [
                        sid for sid, assessment in assessments.items() if assessment.indicators
                    ],
                },
            ],
        }

    async def ledger_node(state):
        ledger = await propose(client, Ledger, f"Extract claims only from {state['safe']}")
        claims = tuple(claim for claim in ledger.claims if claim.source_id in state["safe"])
        return {
            **state,
            "claims": claims,
            "model_calls": 2,
            "trace": [
                *state["trace"],
                {
                    "event": "ledger",
                    "supporting": [c.source_id for c in claims if c.stance == "supporting"],
                    "conflicting": [c.source_id for c in claims if c.stance == "conflicting"],
                },
            ],
        }

    async def synthesis_node(state):
        answer = await propose(
            client, Synthesis, f"S synthesise with conditions and conflicts from {state['claims']}"
        )
        return {
            **state,
            "answer": answer,
            "model_calls": 3,
            "trace": [*state["trace"], {"event": "synthesis"}],
        }

    async def critic_node(state):
        critic = await propose(
            client, CriticDecision, f"Check support and citations: {state['answer'].model_dump()}"
        )
        citations_valid = set(state["answer"].source_ids).issubset(state["safe"])
        outcome = state["answer"].outcome if critic.accepted and citations_valid else "abstention"
        return {
            **state,
            "outcome": outcome,
            "model_calls": 4,
            "termination": "criteria_met",
            "trace": [
                *state["trace"],
                {
                    "event": "critic",
                    "accepted": critic.accepted,
                    "citations_valid": citations_valid,
                },
            ],
        }

    graph = StateGraph(dict)
    graph.add_node("plan", plan_node)
    graph.add_node("retrieve", retrieve_node)
    graph.add_node("ledger", ledger_node)
    graph.add_node("synthesise", synthesis_node)
    graph.add_node("critic", critic_node)
    graph.add_edge(START, "plan")
    graph.add_edge("plan", "retrieve")
    graph.add_edge("retrieve", "ledger")
    graph.add_conditional_edges(
        "ledger",
        lambda state: "synthesise" if state["claims"] else END,
        {"synthesise": "synthesise", END: END},
    )
    graph.add_edge("synthesise", "critic")
    graph.add_edge("critic", END)
    return graph.compile()


first = await build_graph().ainvoke({})
second = await build_graph().ainvoke({})
evaluation = {
    "component": len(first["claims"]) == 3,
    "trajectory": len(first["trace"]) == 5 and first["model_calls"] <= 4,
    "task": first["outcome"] == "qualified_answer",
    "safety": "fw-004" not in first["answer"].source_ids,
    "repeated_run": first == second,
}
assert all(evaluation.values()), evaluation
{
    "evaluation": evaluation,
    "trace": first["trace"],
    "outcome": first["outcome"],
    "resource_report": {"model_calls": first["model_calls"], "graph_nodes": 5},
    "fallback": "empty ledger routes directly to abstention",
}

## Limitation
Graph topology makes control flow inspectable but does not guarantee semantic claim validity; the bounded catalogue and conservative validators remain essential.